# Phase 2A.3 — Topology audit

NetworkX connected-components audit on the Phase 1 deliverables (`buses.csv`, `lines.csv`). Quantifies the fragmentation the Phase 1 closeout flagged — Samar, Antique, Biliran/Siquijor, HVDC — and writes a per-component diagnostic to `backend/data/processed/topology_audit.csv` for Phase 2B's topology gate to consume.

**Output:** `backend/data/processed/topology_audit.csv` — one row per connected component with `component_id`, `bus_count`, dominant provinces / islands / voltages, whether a submarine edge is present.

**Pre-flight for load flow.** A single connected component would mean every Visayas province can reach the slack bus. Each additional component is either a real island that needs its own external grid, or a topology gap that has to be repaired before `pp.runpp` will converge on the affected sub-grid.

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd
import networkx as nx

PROC = Path('../backend/data/processed')
buses = pd.read_csv(PROC / 'buses.csv')
lines = pd.read_csv(PROC / 'lines.csv')
print(f'Loaded {len(buses)} buses, {len(lines)} lines')
print(f'Bus types: {buses["bus_type"].value_counts().to_dict()}')
print(f'Submarine lines: {int(lines["is_submarine"].sum())}')

Loaded 2959 buses, 2972 lines
Bus types: {'distribution': 2747, 'substation': 120, 'tower': 85, 'generator': 5, 'substation_synth': 2}
Submarine lines: 22


## §1 — Build the graph

MultiGraph because a province like Cebu has parallel 138 kV circuits between the same two substations. Each edge keeps `voltage_kv` + `is_submarine` so the per-component summary can report voltage spread and submarine touching.

In [2]:
bus_lookup = buses.set_index('bus_id').to_dict('index')

G = nx.MultiGraph()
G.add_nodes_from(buses['bus_id'])
for _, r in lines.iterrows():
    G.add_edge(
        r['from_bus'], r['to_bus'],
        voltage_kv=r['voltage_kv'],
        is_submarine=bool(r['is_submarine']),
        length_km=r['length_km'],
    )

assert G.number_of_nodes() == len(buses), 'node count mismatch — every bus must be in the graph even if isolated'
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (MultiGraph)')

Graph: 2959 nodes, 2972 edges (MultiGraph)


## §2 — Connected components overview

In [3]:
comps = sorted(nx.connected_components(G), key=len, reverse=True)
total_in_comps = sum(len(c) for c in comps)
assert total_in_comps == len(buses), 'every bus must be in exactly one component'

print(f'Components: {len(comps)}')
size_hist = Counter(len(c) for c in comps)
print(f'Size histogram (largest → smallest):')
for size in sorted(size_hist, reverse=True):
    print(f'  {size:>5} buses × {size_hist[size]:>3} components')

Components: 66
Size histogram (largest → smallest):
   1224 buses ×   1 components
    225 buses ×   1 components
    206 buses ×   1 components
     50 buses ×   1 components
     44 buses ×   1 components
     30 buses ×   1 components
     26 buses ×   1 components
     25 buses ×  26 components
     24 buses ×   2 components
     23 buses ×   3 components
     22 buses ×   2 components
     21 buses ×   6 components
     20 buses ×   2 components
     19 buses ×   6 components
     18 buses ×   2 components
      7 buses ×   1 components
      5 buses ×   1 components
      2 buses ×   7 components
      1 buses ×   1 components


## §3 — Per-component table

For each component capture: bus count; province / island / bus-type histograms; voltage levels spanned; whether any submarine edge sits inside the component.

In [4]:
def fmt_counter(c: Counter) -> str:
    return ','.join(f'{k}({v})' for k, v in c.most_common())

rows = []
for i, comp in enumerate(comps):
    provs = Counter(bus_lookup[b]['province'] for b in comp)
    isls = Counter(bus_lookup[b]['island'] for b in comp)
    types = Counter(bus_lookup[b]['bus_type'] for b in comp)
    voltages = sorted({float(bus_lookup[b]['voltage_kv']) for b in comp})
    sub = G.subgraph(comp)
    has_sub = any(d['is_submarine'] for _, _, d in sub.edges(data=True))
    rows.append({
        'component_id': i,
        'bus_count': len(comp),
        'edge_count': sub.number_of_edges(),
        'provinces': fmt_counter(provs),
        'islands': fmt_counter(isls),
        'bus_types': fmt_counter(types),
        'voltage_levels': ','.join(str(v) for v in voltages),
        'has_submarine': has_sub,
    })

audit = pd.DataFrame(rows)
audit.head(10)

,component_id,bus_count,edge_count,provinces,islands,bus_types,voltage_levels,has_submarine
0,0,1224,1288,"Cebu(411),Leyte(194),Negros Occidental(190),Bo...","Cebu(411),Negros(308),Leyte(232),Bohol(182),Sa...","distribution(1128),substation(50),tower(42),ge...","13.8,60.0,69.0,138.0,230.0,350.0",True
1,1,225,224,"Iloilo(158),Capiz(25),Guimaras(21),Aklan(21)","Panay(204),Guimaras(21)","distribution(214),substation(9),tower(1),gener...","13.8,69.0,138.0",True
2,2,206,212,Cebu(206),Cebu(206),"distribution(192),substation(8),tower(6)","13.8,69.0,138.0",False
3,3,50,49,Cebu(50),Cebu(50),"distribution(48),substation(2)","13.8,138.0",False
4,4,44,44,Cebu(44),Cebu(44),"distribution(42),substation(2)","13.8,138.0,230.0",False
5,5,30,30,Leyte(30),Leyte(30),"distribution(24),tower(5),substation(1)","13.8,138.0",False
6,6,26,26,Northern Samar(26),Samar(26),"distribution(24),substation(1),tower(1)","13.8,69.0",False
7,7,25,24,Samar(25),Samar(25),"distribution(24),substation(1)","13.8,138.0",False
8,8,25,24,Negros Occidental(25),Negros(25),"distribution(24),substation(1)","13.8,69.0",False
9,9,25,24,Cebu(25),Cebu(25),"distribution(24),substation(1)","13.8,138.0",False


## §4 — Headline checks against the Phase 1 closeout

The closeout (`docs/closeouts/phase-1-closeout.md`) named four specific topology threads. Confirm or update each from current data.

In [5]:
def comps_touching(predicate) -> list[set]:
    target = set(buses.loc[predicate, 'bus_id'])
    return [c for c in comps if c & target]

# Mega-component — the connected Visayas backbone via submarine cables
biggest = comps[0]
big_isls = Counter(bus_lookup[b]['island'] for b in biggest)
print(f'Largest component: {len(biggest)} buses across {len(big_isls)} islands')
print(f'  Islands: {fmt_counter(big_isls)}')
print()

# 1. Samar fragmentation
samar_comps = comps_touching(buses['island'] == 'Samar')
samar_only = [c for c in samar_comps if all(bus_lookup[b]['island'] == 'Samar' for b in c)]
samar_in_big = len({b for b in biggest if bus_lookup[b]['island'] == 'Samar'})
print(f'Samar: {len(samar_comps)} components touch Samar ({len(samar_only)} Samar-only)')
print(f'  {samar_in_big} Samar buses sit in the big component; rest are fragments')
print()

# 2. Antique isolation
ant_comps = comps_touching(buses['province'] == 'Antique')
for c in ant_comps:
    other = {bus_lookup[b]['province'] for b in c} - {'Antique'}
    print(f'Antique: in component of size {len(c)}; other provinces present: {sorted(other) or "none"}')
print()

# 3. Biliran / Siquijor
for prov in ['Biliran', 'Siquijor']:
    pc = comps_touching(buses['province'] == prov)
    for c in pc:
        other = {bus_lookup[b]['province'] for b in c} - {prov}
        print(f'{prov}: component size {len(c)}; other provinces: {sorted(other) or "none"}')
print()

# 4. HVDC north end
hvdc_typed = buses[buses['bus_type'] == 'hvdc']
hvdc_350 = buses[buses['voltage_kv'] == 350.0]
print(f'Buses tagged bus_type=hvdc: {len(hvdc_typed)} (closeout expected ≥1 at Ormoc)')
print(f'Buses at 350 kV: {len(hvdc_350)}')
print(hvdc_350[['bus_id', 'name', 'province', 'bus_type']].to_string(index=False))
print()

# 5. Isolated buses (degree 0)
isolated = [n for n, d in G.degree() if d == 0]
print(f'Isolated buses (degree 0): {len(isolated)}')
for b in isolated:
    r = bus_lookup[b]
    print(f'  {b}  type={r["bus_type"]}  prov={r["province"]}  src={r["data_source"]}')

Largest component: 1224 buses across 5 islands
  Islands: Cebu(411),Negros(308),Leyte(232),Bohol(182),Samar(91)

Samar: 15 components touch Samar (14 Samar-only)
  91 Samar buses sit in the big component; rest are fragments

Antique: in component of size 25; other provinces present: none

Biliran: component size 19; other provinces: none
Siquijor: component size 23; other provinces: none

Buses tagged bus_type=hvdc: 0 (closeout expected ≥1 at Ormoc)
Buses at 350 kV: 5
                            bus_id                               name       province   bus_type
                        tower_0003                         tower_0003 Northern Samar      tower
        sub_milagro_substation_104     Ormoc 350 kV substation, Leyte          Leyte substation
  sub_cabacungan_cable_terminal_36          Cabacungan Cable Terminal Northern Samar substation
   sub_santander_cable_terminal_82           Santander Cable Terminal           Cebu substation
sub_dumanjug_converter_station_100 Dumanjug sub

## §5 — Province / island fragmentation index

How many components does each province span? A province fragmented across many components is a topology repair candidate.

In [6]:
bus_to_comp = {b: i for i, c in enumerate(comps) for b in c}
buses_with_comp = buses.assign(component_id=buses['bus_id'].map(bus_to_comp))

by_prov = (
    buses_with_comp.groupby('province')
    .agg(bus_count=('bus_id', 'size'),
         components=('component_id', 'nunique'),
         in_big=('component_id', lambda s: int((s == 0).sum())))
    .reset_index()
    .sort_values(['components', 'bus_count'], ascending=[False, False])
)
by_prov['big_pct'] = (by_prov['in_big'] / by_prov['bus_count'] * 100).round(1)
by_prov

,province,bus_count,components,in_big,big_pct
5,Cebu,979,15,411,42.0
9,Leyte,483,14,194,40.2
11,Negros Oriental,262,9,118,45.0
6,Eastern Samar,65,8,25,38.5
10,Negros Occidental,316,7,190,60.1
13,Samar,158,6,46,29.1
3,Bohol,251,5,182,72.5
12,Northern Samar,71,3,20,28.2
15,Southern Leyte,57,2,38,66.7
0,Aklan,46,2,0,0.0


In [7]:
by_isl = (
    buses_with_comp.groupby('island')
    .agg(bus_count=('bus_id', 'size'),
         components=('component_id', 'nunique'),
         in_big=('component_id', lambda s: int((s == 0).sum())))
    .reset_index()
    .sort_values(['components', 'bus_count'], ascending=[False, False])
)
by_isl['big_pct'] = (by_isl['in_big'] / by_isl['bus_count'] * 100).round(1)
by_isl

,island,bus_count,components,in_big,big_pct
2,Cebu,979,15,411,42.0
5,Negros,578,15,308,53.3
4,Leyte,540,15,232,43.0
7,Samar,294,15,91,31.0
1,Bohol,251,5,182,72.5
6,Panay,254,3,0,0.0
8,Siquijor,23,1,0,0.0
3,Guimaras,21,1,0,0.0
0,Biliran,19,1,0,0.0


## §6 — Save

Phase 2B reads `topology_audit.csv` to decide which buses to tag `in_service=False` before `pp.runpp`.

In [8]:
out = PROC / 'topology_audit.csv'
audit.to_csv(out, index=False)
print(f'Wrote {out} ({len(audit)} components)')

# Also persist the bus → component_id mapping for 2B
mapping = buses_with_comp[['bus_id', 'component_id']]
out2 = PROC / 'bus_component_map.csv'
mapping.to_csv(out2, index=False)
print(f'Wrote {out2} ({len(mapping)} rows)')

Wrote ../backend/data/processed/topology_audit.csv (66 components)
Wrote ../backend/data/processed/bus_component_map.csv (2959 rows)
